# Update Databricks Secrets ACLs

> **Provenance:** Cloned from `helper-notebooks/Update Databricks Secrets ACLs`
> (workspace: `fevm-hls-fde.cloud.databricks.com`).  
> Original source: [matthew-giglia/databricks-helper-notebooks](https://github.com/matthew-giglia/databricks-helper-notebooks)

Use this notebook to grant or revoke READ/WRITE/MANAGE permissions on a
secret scope for service principals, groups, or users. The Databricks App's
auto-provisioned SPN needs READ access to the `lakeloom_credentials` scope —
run this notebook after the App bundle deploys to grant that access.

---


In [0]:
%pip install databricks-sdk --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.8/832.8 kB 37.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-eddcf73a-39c5-4782-8b8a-78b335895759
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.67.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-eddcf73a-39c5-4782-8b8a-78b335895759
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("secret_scope", "", "Databricks Secret Scope")
dbutils.widgets.text("principal", "", "Service Principal, Group, or User")
dbutils.widgets.dropdown("permission", "READ", ["READ", "WRITE", "MANAGE", "REVOKE"], "ACL Permission")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import workspace

w = WorkspaceClient()

In [0]:
secret_scope = dbutils.widgets.get("secret_scope")
principal = dbutils.widgets.get("principal")
permission = dbutils.widgets.get("permission")

print(
f"""
   secret_scope: {secret_scope}
   principal: {principal}
   permission: {permission}   
""")


   secret_scope: fhir_zerobus_credentials
   principal: cd633c8e-d38c-413a-bcf5-21fc3e9f2840
   permission: READ   



In [0]:
scopes = w.secrets.list_scopes()
scopes = [scope.as_dict() for scope in scopes]
scope_exists = any(scope['name'] == secret_scope for scope in scopes)
print(f"scope_exists: {scope_exists}")

if scope_exists == True:
  acls = w.secrets.list_acls(scope=secret_scope)
  acls = [acl.as_dict() for acl in acls]
  print(acls)
else:
  raise Exception("Secret Scope does not exist, please check the input widget or create a new secret scope with the 'Set Databricks Secrets with Python SDK' notebook.")

scope_exists: True
[{'permission': 'READ', 'principal': '85f31ec8-d13a-45cd-be6b-dadef4e2e1c0'}, {'permission': 'READ', 'principal': '1918013d-90e8-4e78-b2e9-7596ca2dbfae'}, {'permission': 'MANAGE', 'principal': 'matthew.giglia@databricks.com'}, {'permission': 'MANAGE', 'principal': 'acf021b4-87c6-44ff-b3d7-45c59d63fe4d'}]


In [0]:
if permission == "REVOKE":
    for acl in acls: 
        if acl['principal'] == principal: 
            w.secrets.delete_acl(scope=secret_scope, principal=acl['principal'])
            print(f"Revoking access to Databricks Secret Scope '{secret_scope}' for '{principal}'.")
        else:
            pass
else:
    w.secrets.put_acl(
        scope = secret_scope
        ,principal = principal
        ,permission = workspace.AclPermission[permission]
    )
    print(f"Granting `{permission}` access to Databricks Secret Scope '{secret_scope}' for '{principal}'.")

---------------------------------------------------------------------------
InvalidParameterValue                     Traceback (most recent call last)
File <command-8502073128475179>, line 9
      7             pass
      8 else:
----> 9     w.secrets.put_acl(
     10         scope = secret_scope
     11         ,principal = principal
     12         ,permission = workspace.AclPermission[permission]
     13     )
     14     print(f"Granting `{permission}` access to Databricks Secret Scope '{secret_scope}' for '{principal}'.")

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-eddcf73a-39c5-4782-8b8a-78b335895759/lib/python3.12/site-packages/databricks/sdk/service/workspace.py:2647, in SecretsAPI.put_acl(self, scope, principal, permission)
   2644 if cfg.host_type == HostType.UNIFIED and cfg.workspace_id:
   2645     headers["X-Databricks-Org-Id"] = cfg.workspace_id
-> 2647 self._api.do("POST", "/api/2.0/secrets/acls/put", body=body, headers=headers)

File /local_disk0/.ephemeral_nfs/en

In [0]:
acls = w.secrets.list_acls(scope=secret_scope)
acls = [acl.as_dict() for acl in acls]
print(acls)